# CellIQ tissue model: head-to-head test of the "v2 grayscale-robust" recipe

**Question:** how does the training recipe in `CellIQ_lung_training_v2.ipynb` (random split, 50% grayscale, label smoothing, cosine schedule, 4 + 8 epochs)
perform next to our own tissue model (`tissue_v2`, near-duplicate-aware split)?

It runs that recipe **twice** on the LC25000 lung slides and reports the same checks our own model went through:

| Run | Split | Why |
|---|---|---|
| **A. as written** | random split, exactly like the v2 notebook | shows what score that notebook would report |
| **B. fair test** | our near-duplicate-aware split (same test images as our model) | head-to-head with our model |

For each run: accuracy, accuracy with colour removed (grayscale), the cancer-vs-normal check (risk = 1 - P(normal), threshold picked on the validation set),
and how many test images have a near-identical twin in the training set (leakage).

**How to run:** Runtime > Change runtime type > T4 GPU, then Runtime > Run all. About 30 to 40 minutes.

Research and educational prototype. Not for real diagnosis.

## 1. Settings

In [ ]:
DATASET = "lung_slides"
IMG_SIZE = 224
BATCH_SIZE = 32
SEED = 42
OUT_DIR = "/content/compare_out"

# The recipe from CellIQ_lung_training_v2.ipynb
GRAY_PROB = 0.5
LABEL_SMOOTH = 0.1
HEAD_EPOCHS = 4
FINETUNE_EPOCHS = 8

# Our own tissue_v2 model, measured earlier on the near-duplicate-aware split (models/tissue_v2/metrics.json)
OUR = {"accuracy": 0.9924, "accuracy_grayscale": 0.9877, "n_test": 1711,
       "sensitivity": 0.997, "specificity": 1.0, "auc": 1.0,
       "twin_rate_at_0.90": 0.0, "twin_rate_at_0.85": 0.4974}

## 2. Tools, data download and file list (same code as our own tissue notebook)

In [ ]:
import os, json, random, re, shutil
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device)
os.makedirs(OUT_DIR, exist_ok=True)
from sklearn.model_selection import train_test_split
from concurrent.futures import ThreadPoolExecutor

In [ ]:
import kagglehub
path = kagglehub.dataset_download("javaidahmadwani/lc25000")
DATA_DIR = None
for root, dirs, _ in os.walk(path):
    if {"lung_aca", "lung_n", "lung_scc"} <= set(dirs):
        DATA_DIR = root
        break
print("DATA_DIR =", DATA_DIR)
assert DATA_DIR, "lung folders not found"

In [ ]:
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp")
class_names = sorted([d for d in os.listdir(DATA_DIR)
                      if os.path.isdir(os.path.join(DATA_DIR, d)) and d.startswith("lung_")])
print("Groups:", class_names)
natural = lambda s: [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", s)]
files, labels = [], []
for idx, d in enumerate(class_names):
    folder = os.path.join(DATA_DIR, d)
    for f in sorted(os.listdir(folder), key=natural):
        if f.lower().endswith(IMG_EXT):
            files.append(os.path.join(folder, f)); labels.append(idx)
files = np.array(files); labels = np.array(labels)
print("Total images:", len(files), "| per group:", np.bincount(labels))

## 3. Load every image once at 224 x 224 (same resize as the recipe uses first, so pixels are identical, but training is much faster)

In [ ]:
def load_resized(path):
    return np.asarray(Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR))

with ThreadPoolExecutor(max_workers=8) as ex:
    CACHE = np.stack(list(ex.map(load_resized, files)))
print("cached images:", CACHE.shape, "| %.1f GB" % (CACHE.nbytes / 1e9))

## 4. Group near-duplicates and split by group (our method; reused unchanged)

In [ ]:
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
emb_tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), transforms.Normalize(mean, std)])

class PlainDS(Dataset):
    def __len__(self): return len(files)
    def __getitem__(self, i): return emb_tf(Image.fromarray(CACHE[i])), 0

emb = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT).to(device).eval()
emb.classifier = nn.Identity()
feats = []
with torch.no_grad():
    for x, _ in DataLoader(PlainDS(), batch_size=64, num_workers=2):
        feats.append(emb(x.to(device)).float().cpu())
feats = torch.cat(feats)
feats = F.normalize(feats - feats.mean(0), dim=1).to(device)
N = len(feats)

FLOOR = 0.75
pi, pj, ps = [], [], []
for i in range(0, N, 1024):
    s = feats[i:i+1024] @ feats.T
    r, c = torch.where(s >= FLOOR)
    pi.append((r + i).cpu().numpy()); pj.append(c.cpu().numpy()); ps.append(s[r, c].cpu().numpy())
pi, pj, ps = np.concatenate(pi), np.concatenate(pj), np.concatenate(ps)
print("Linked pairs at similarity >=", FLOOR, ":", len(pi))

def components(thr):
    m = ps >= thr
    G = coo_matrix((np.ones(m.sum()), (pi[m], pj[m])), shape=(N, N))
    n, gid = connected_components(G, directed=False)
    return n, gid

table = {}
for thr in (0.80, 0.825, 0.85, 0.875, 0.90, 0.925, 0.95):
    n, gid = components(thr)
    sizes = np.bincount(gid)
    table[thr] = (n, gid, int(sizes.max()))
    print(f"similarity >= {thr:.3f}: groups={n:5d} | biggest group={sizes.max():5d} | median size={int(np.median(sizes))}")

# Linking images by similarity can chain unrelated photos into one giant group (which breaks the split).
# Use the LOWEST threshold (most careful grouping) whose biggest group is still small enough to split fairly.
MAX_GROUP = 700
ok_thr = [t for t in table if table[t][2] <= MAX_GROUP]
SIM = min(ok_thr) if ok_thr else max(table)
n_groups, group_id, _ = table[SIM]
print("Chosen similarity threshold:", SIM, "->", n_groups, "groups")

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED)
folds = [idx for _, idx in sgkf.split(files, labels, group_id)]
te, va = folds[0], folds[1]
tr = np.concatenate(folds[2:])
for nm, ix in (("train", tr), ("val", va), ("test", te)):
    print(f"{nm:5s} n={len(ix):5d} per group: {np.bincount(labels[ix], minlength=len(class_names))}")

# leakage check: how many test images still have a similar-looking twin in train? (random split vs group split)
def twin_rates(test_idx, train_idx, thrs=(0.95, 0.90, 0.85, 0.80)):
    best = []
    for i in range(0, len(test_idx), 512):
        s = feats[test_idx[i:i+512]] @ feats[train_idx].T
        best.append(s.max(1).values.cpu().numpy())
    best = np.concatenate(best)
    return {str(t): round(float((best >= t).mean()), 4) for t in thrs}

rng = np.random.RandomState(SEED); perm = rng.permutation(N)
r_te, r_tr = perm[:len(te)], perm[len(te) + len(va):]
leak_random = twin_rates(r_te, r_tr)
leak_group = twin_rates(te, tr)
print("Share of test images with a near-identical twin in train, at similarity >= 0.95 / 0.90 / 0.85 / 0.80:")
print("  random split:", leak_random)
print("  group split :", leak_group, "(zero at or above the grouping threshold by construction; lower levels show what remains)")

In [ ]:
print("our group-aware test set size:", len(te), "(our saved model was tested on", OUR["n_test"], "images)")

## 5. The v2 recipe: transforms, training and the checks

In [ ]:
their_train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomGrayscale(p=GRAY_PROB),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.25, contrast=0.25),
    transforms.ToTensor(), transforms.Normalize(mean, std)])
their_eval_tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), transforms.Normalize(mean, std)])
their_gray_tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.Grayscale(3),
                                    transforms.ToTensor(), transforms.Normalize(mean, std)])
K = len(class_names)


class CachedDS(Dataset):
    def __init__(self, idxs, tf):
        self.idxs, self.tf = idxs, tf
    def __len__(self):
        return len(self.idxs)
    def __getitem__(self, i):
        j = self.idxs[i]
        return self.tf(Image.fromarray(CACHE[j])), int(labels[j])


def train_their_recipe(tr_idx, va_idx, tag):
    torch.manual_seed(SEED)
    mdl = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    mdl.classifier[1] = nn.Linear(mdl.classifier[1].in_features, K)
    mdl = mdl.to(device)
    tdl = DataLoader(CachedDS(tr_idx, their_train_tf), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    vdl = DataLoader(CachedDS(va_idx, their_eval_tf), batch_size=BATCH_SIZE, num_workers=2)
    cnt = np.bincount(labels[tr_idx], minlength=K)
    w = torch.tensor(cnt.sum() / (K * cnt), dtype=torch.float32).to(device)
    loss_fn = nn.CrossEntropyLoss(weight=w, label_smoothing=LABEL_SMOOTH)
    best = {"loss": 1e9, "state": None}

    def run_epoch(dl, train, opt=None):
        mdl.train(train)
        total, correct, loss_sum = 0, 0, 0.0
        with torch.set_grad_enabled(train):
            for x, y in dl:
                x, y = x.to(device), y.to(device)
                out = mdl(x)
                loss = loss_fn(out, y)
                if train:
                    opt.zero_grad(); loss.backward(); opt.step()
                loss_sum += loss.item() * len(y); total += len(y)
                correct += (out.argmax(1) == y).sum().item()
        return loss_sum / total, correct / total

    def fit(epochs, lr, name):
        opt = torch.optim.AdamW(filter(lambda p: p.requires_grad, mdl.parameters()), lr=lr)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
        for ep in range(epochs):
            tl, ta = run_epoch(tdl, True, opt)
            vl, va_ = run_epoch(vdl, False)
            print(f"[{tag}] {name} epoch {ep + 1}: train loss {tl:.3f} acc {ta:.3f} | val loss {vl:.3f} acc {va_:.3f}", flush=True)
            sched.step()
            if vl < best["loss"]:
                best["loss"] = vl
                best["state"] = {k: v.detach().cpu().clone() for k, v in mdl.state_dict().items()}

    for p in mdl.parameters():
        p.requires_grad = False
    for p in mdl.classifier.parameters():
        p.requires_grad = True
    fit(HEAD_EPOCHS, 1e-3, "phase 1 (head only)")
    for p in mdl.parameters():
        p.requires_grad = True
    fit(FINETUNE_EPOCHS, 1e-4, "phase 2 (fine-tune)")
    mdl.load_state_dict(best["state"])
    return mdl


@torch.no_grad()
def get_probs(mdl, idx, tf):
    mdl.eval()
    P, Y = [], []
    for x, y in DataLoader(CachedDS(idx, tf), batch_size=64, num_workers=2):
        P.append(F.softmax(mdl(x.to(device)), dim=1).cpu().numpy())
        Y.append(y.numpy())
    return np.concatenate(P), np.concatenate(Y)


def evaluate(mdl, te_idx, va_idx):
    """The same checks our own tissue model went through."""
    pt, yt = get_probs(mdl, te_idx, their_eval_tf)
    pg, _ = get_probs(mdl, te_idx, their_gray_tf)
    pv, yv = get_probs(mdl, va_idx, their_eval_tf)
    pred, pred_g = pt.argmax(1), pg.argmax(1)
    cm = confusion_matrix(yt, pred)
    n_idx = class_names.index("lung_n")
    risk_v, risk_t = 1 - pv[:, n_idx], 1 - pt[:, n_idx]
    yb_v, yb_t = (yv != n_idx).astype(int), (yt != n_idx).astype(int)
    cands = np.linspace(0.01, 0.99, 99)
    ok = [t for t in cands if (risk_v[yb_v == 1] >= t).mean() >= 0.95]      # highest threshold that keeps 95% sensitivity on validation
    thr = float(max(ok)) if ok else 0.5
    tn, fp, fn, tp = confusion_matrix(yb_t, (risk_t >= thr).astype(int), labels=[0, 1]).ravel()
    return {"n_test": int(len(te_idx)), "accuracy": round(float((pred == yt).mean()), 4),
            "accuracy_grayscale": round(float((pred_g == yt).mean()), 4), "wrong": int((pred != yt).sum()),
            "sensitivity": round(float(tp / max(tp + fn, 1)), 3), "specificity": round(float(tn / max(tn + fp, 1)), 3),
            "auc": round(float(roc_auc_score(yb_t, risk_t)), 3), "threshold": round(thr, 3),
            "confusion_matrix": cm.tolist()}
print("ready")

## 6. Run A: the v2 notebook exactly as written (random split)

In [ ]:
idx_all = np.arange(N)
a_tr, tmp = train_test_split(idx_all, test_size=0.30, stratify=labels, random_state=SEED)
a_va, a_te = train_test_split(tmp, test_size=0.50, stratify=labels[tmp], random_state=SEED)
print("run A sizes (train / val / test):", len(a_tr), len(a_va), len(a_te))
model_A = train_their_recipe(a_tr, a_va, "A random split")
res_A = evaluate(model_A, a_te, a_va)
res_A["twin_rate"] = twin_rates(a_te, a_tr)
print("RUN A (random split):", {k: v for k, v in res_A.items() if k != "confusion_matrix"})
del model_A
torch.cuda.empty_cache()

## 7. Run B: the same recipe on our near-duplicate-aware split (the fair test)

In [ ]:
model_B = train_their_recipe(tr, va, "B group split")
res_B = evaluate(model_B, te, va)
res_B["twin_rate"] = twin_rates(te, tr)
print("RUN B (group split):", {k: v for k, v in res_B.items() if k != "confusion_matrix"})
del model_B
torch.cuda.empty_cache()

## 8. Side-by-side table and save

In [ ]:
rows = [("Test images", "n_test"), ("Accuracy", "accuracy"), ("Accuracy, colour removed", "accuracy_grayscale"),
        ("Cancer vs normal: sensitivity", "sensitivity"), ("Cancer vs normal: specificity", "specificity"), ("Cancer vs normal: AUC", "auc")]
print(f"{'':36s}{'A: v2 as written':>20s}{'B: v2 on our split':>22s}{'Our tissue_v2':>16s}")
for label, key in rows:
    print(f"{label:36s}{res_A[key]:>20}{res_B[key]:>22}{OUR[key]:>16}")
print(f"{'Wrong test predictions':36s}{res_A['wrong']:>20}{res_B['wrong']:>22}{'':>16}")
print(f"{'Test images with a twin in train':36s}{'(similarity >= 0.90 / 0.85)':>20}")
print(f"{'   run A (random split)':36s}{res_A['twin_rate']['0.9']:>20}{res_A['twin_rate']['0.85']:>22}")
print(f"{'   run B / ours (group split)':36s}{res_B['twin_rate']['0.9']:>20}{res_B['twin_rate']['0.85']:>22}")

os.makedirs(OUT_DIR, exist_ok=True)
out = {"our_tissue_v2": OUR, "run_A_v2_as_written_random_split": res_A, "run_B_v2_recipe_on_group_split": res_B,
       "note": "Same EfficientNet-B0 and data. A uses a random split (near-duplicates can sit in train and test), B uses the near-duplicate-aware split. Single run each: differences of a few images are within noise. One public dataset, not clinically validated."}
json.dump(out, open(f"{OUT_DIR}/compare_results.json", "w"), indent=1)
shutil.make_archive("/content/cellIQ_tissue_compare", "zip", OUT_DIR)
print("zip ready:", os.path.getsize("/content/cellIQ_tissue_compare.zip"), "bytes")